In [1]:
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

print("Imports completed.")

Imports completed.


In [ ]:
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

NOTEBOOK_PATH = Path.cwd()
PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

cleaned_file = PROCESSED_DIR / "AIS_2024_01_panynj_cleaned.csv"

print("Loading cleaned dataset...")
print("File exists:", cleaned_file.exists())

AIS_cleaned = pd.read_csv(cleaned_file)

print("\nDataset loaded.")
display(AIS_cleaned.shape)
display(AIS_cleaned.head())


Loading cleaned dataset...
File exists: True


In [ ]:
print("Dataset structural check...")
print("=" * 40)

print("Shape:")
display(AIS_cleaned.shape)
print("=" * 40)
print("\nColumns:")
display(AIS_cleaned.columns.tolist())
print("=" * 40)
print("\nData types:")
display(AIS_cleaned.dtypes)
print("=" * 40)
print("\nMemory usage (MB):")
display(round(AIS_cleaned.memory_usage(deep=True).sum() / 1_000_000, 2))
print("=" * 40)
print("\nSample rows:")
display(AIS_cleaned.head())
print("=" * 40)
print("\nDataset description check completed.")

In [ ]:
core_cols = [
    "MMSI",
    "Day",
    "TimeOfDay",
]

movement_cols = [
    "LAT",
    "LON",
    "SOG",
    "COG",
    "Heading",
    "Status",
]

behaviour_cols = core_cols + movement_cols

AIS_movement = AIS_cleaned.loc[:, behaviour_cols].copy()

print(f"Behavioural working dataset created.\n{"=" * 40}")
print(f"Core columns: {core_cols}\n{"=" * 40}")
print(f"Movement columns:{movement_cols}\n{"=" * 40}")
print(f"Total selected columns: {len(behaviour_cols)}\n{"=" * 40}")
print("AIS_movement shape:", AIS_movement.shape)

display(AIS_movement.head())

In [ ]:
print(f"Ordering dataset by MMSI, Day, and TimeOfDay...\n{'=' * 40}")

AIS_movement = (
    AIS_movement
    .sort_values(
        by=["MMSI", "Day", "TimeOfDay"],
        ascending=[True, True, True],
        kind="mergesort"  # stable sort — preserves order where values are equal
    )
    .reset_index(drop=True)
)

print(f"Dataset ordered successfully.\n{'=' * 40}")
print("AIS_movement shape:", AIS_movement.shape)

display(AIS_movement.head())

In [ ]:
#this step was added in order to calculate timedelta later
print("Step 1 — Creating unified timestamp.\n")

AIS_movement["Timestamp"] = pd.to_datetime(
    "2024-01-" 
    + AIS_movement["Day"].astype(str).str.zfill(2)
    + " "
    + AIS_movement["TimeOfDay"].astype(str),
    errors="coerce"
)

print("Timestamp column created.")

display(
    AIS_movement[
        ["Day", "TimeOfDay", "Timestamp"]
    ].head()
)

print("\nNull timestamps:", AIS_movement["Timestamp"].isna().sum())

In [ ]:
print("Step 2 — Creating previous-record reference fields.\n")

AIS_movement["prev_timestamp"] = (
    AIS_movement
    .groupby("MMSI")["Timestamp"]
    .shift(1)
)

AIS_movement["prev_LAT"] = (
    AIS_movement
    .groupby("MMSI")["LAT"]
    .shift(1)
)

AIS_movement["prev_LON"] = (
    AIS_movement
    .groupby("MMSI")["LON"]
    .shift(1)
)

print("Previous-record fields created.")

display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "prev_timestamp",
            "LAT",
            "prev_LAT",
            "LON",
            "prev_LON",
        ]
    ].head()
)

print("\nRows without previous timestamp:", AIS_movement["prev_timestamp"].isna().sum())

In [ ]:
print("Step 3 — Calculating time difference between transmissions.\n")

AIS_movement["time_delta_seconds"] = (
    AIS_movement["Timestamp"]
    - AIS_movement["prev_timestamp"]
).dt.total_seconds()

print("Time delta calculation complete.")

display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "prev_timestamp",
            "time_delta_seconds",
        ]
    ].head()
)

print(
    "\nNegative time deltas:",
    (AIS_movement["time_delta_seconds"] < 0).sum()
)

## Movement Distance Calculation — Method Selection

A simple subtraction of latitude and longitude values is **not sufficient** to measure vessel movement.

Coordinates are angular positions on the Earth's surface, not linear distances.  
Therefore, a change in latitude or longitude does not directly represent a physical distance.

To calculate movement between consecutive AIS transmissions, the **Haversine formula** is used.  
This method accounts for the curvature of the Earth and produces a physically meaningful distance.

The formula used is:

a = sin²(Δlat / 2) + cos(lat₁) × cos(lat₂) × sin²(Δlon / 2)  
c = 2 × atan2( √a , √(1 − a) )  
distance = R × c  

Where:

- lat₁, lon₁ = previous position  
- lat₂, lon₂ = current position  
- Δlat = lat₂ − lat₁  
- Δlon = lon₂ − lon₁  
- R = Earth's radius (6,371,000 meters)

Distance is calculated in:

**meters**

rather than knots because:

- time differences are measured in **seconds**
- movement rate calculations require consistent units
- meters provide a standard, system-independent measurement

The resulting column:

**distance_m**

represents the physical distance travelled between the current and previous transmission for the same vessel.

Expected behaviour:

- The first record for each vessel will have `NaN` distance  
- This is correct, as no previous position exists for comparison  

---

## References

NOAA — AIS Data and Vessel Tracking  
https://coast.noaa.gov/

Movable Type Scripts — Haversine Distance Explanation  
https://www.movable-type.co.uk/scripts/latlong.html

International Maritime Organization (IMO) — AIS Operational Principles  
https://www.imo.org/

National Geospatial-Intelligence Agency — Geodesy Reference  
https://earth-info.nga.mil/

In [ ]:

print("Step 4 — Calculating movement distance (meters).\n")

# Earth radius in meters
EARTH_RADIUS_M = 6_371_000

# Convert degrees to radians
lat1 = np.radians(AIS_movement["prev_LAT"])
lon1 = np.radians(AIS_movement["prev_LON"])

lat2 = np.radians(AIS_movement["LAT"])
lon2 = np.radians(AIS_movement["LON"])

# Haversine formula
dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

c = 2 * np.arctan2(
    np.sqrt(a),
    np.sqrt(1 - a)
)

AIS_movement["distance_m"] = EARTH_RADIUS_M * c

print("Movement distance calculation complete.")

display(
    AIS_movement[
        [
            "MMSI",
            "prev_LAT",
            "prev_LON",
            "LAT",
            "LON",
            "distance_m",
        ]
    ].head()
)

print(
    "\nRows with missing distance:",
    AIS_movement["distance_m"].isna().sum()
)

In [ ]:
print("Step 5 — Calculating vessel speed (m/s).\n")

# Speed = Distance / Time

AIS_movement["speed_mps"] = (
    AIS_movement["distance_m"] /
    AIS_movement["time_delta_seconds"]
)

print("Speed calculation complete.")



In [ ]:
AIS_movement["speed_mps"] = AIS_movement["speed_mps"].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite speed values:", np.isinf(AIS_movement["speed_mps"]).sum())
print("Missing speed values:", AIS_movement["speed_mps"].isna().sum())

In [ ]:
display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "distance_m",
            "time_delta_seconds",
            "speed_mps"
        ]
    ].head()
)

print("\nNull speeds:", AIS_movement["speed_mps"].isna().sum())
print("Negative speeds:", (AIS_movement["speed_mps"] < 0).sum())
print("Infinite speeds:", np.isinf(AIS_movement["speed_mps"]).sum())

In [ ]:
print("Step 6.1 — Creating previous speed reference.\n")

AIS_movement["prev_speed_mps"] = (
    AIS_movement
        .groupby("MMSI")["speed_mps"]
        .shift(1)
)

print("Previous speed reference created.")

display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "speed_mps",
            "prev_speed_mps"
        ]
    ].head()
)

print("\nRows without previous speed:",
      AIS_movement["prev_speed_mps"].isna().sum())

In [ ]:
print("Step 6.2 — Calculating vessel acceleration (m/s²).\n")

# Acceleration = change in speed / time

AIS_movement["acceleration_mps2"] = (
    (
        AIS_movement["speed_mps"]
        -
        AIS_movement["prev_speed_mps"]
    )
    /
    AIS_movement["time_delta_seconds"]
)

print("Acceleration calculation complete.")



In [ ]:
AIS_movement["acceleration_mps2"] = AIS_movement["acceleration_mps2"].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite acceleration values:", np.isinf(AIS_movement["acceleration_mps2"]).sum())
print("Missing acceleration values:", AIS_movement["acceleration_mps2"].isna().sum())

In [ ]:
display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "speed_mps",
            "prev_speed_mps",
            "time_delta_seconds",
            "acceleration_mps2"
        ]
    ].head()
)

print("\nNull accelerations:",
      AIS_movement["acceleration_mps2"].isna().sum())

print("Negative accelerations:",
      (AIS_movement["acceleration_mps2"] < 0).sum())

print("Infinite accelerations:",
      np.isinf(AIS_movement["acceleration_mps2"]).sum())

## Step 7 — Movement State Classification

Vessels are classified into behaviour bands based on calculated speed (m/s).

Thresholds are based on typical maritime operational ranges:

| State | Speed Range | Approximate Knots | Interpretation |
|---|---|---|---|
| UNKNOWN | null | — | Missing speed — first row per vessel or post-null propagation |
| STATIONARY | 0 m/s | 0 kn | No movement |
| VERY_SLOW | 0–0.5 m/s | < 1 kn | Minimal drift, manoeuvring at berth |
| SLOW | 0.5–3 m/s | 1–6 kn | Harbour manoeuvring, approach/departure |
| NORMAL | 3–10 m/s | 6–19 kn | Coastal/channel transit |
| HIGH_SPEED | > 10 m/s | > 19 kn | Open water transit, fast ferries |

These thresholds are approximate. They are suitable for grouping and congestion analysis but are not calibrated against AIS Status codes or external ground truth.

Note: `speed_mps` is derived from GPS position changes (Haversine distance / time delta). It differs from the AIS-reported `SOG` field, which is the vessel's own speed log reading. Both are available for comparison in the analysis phase.

In [ ]:
print("Step 7 — Classifying vessel movement behaviour.\n")

conditions = [
    AIS_movement["speed_mps"].isna(),
    AIS_movement["speed_mps"] == 0,
    (AIS_movement["speed_mps"] > 0) & (AIS_movement["speed_mps"] <= 0.5),
    (AIS_movement["speed_mps"] > 0.5) & (AIS_movement["speed_mps"] <= 3),
    (AIS_movement["speed_mps"] > 3) & (AIS_movement["speed_mps"] <= 10),
    AIS_movement["speed_mps"] > 10
]

labels = [
    "UNKNOWN",
    "STATIONARY",
    "VERY_SLOW",
    "SLOW",
    "NORMAL",
    "HIGH_SPEED"
]

AIS_movement["movement_state"] = np.select(
    conditions,
    labels,
    default="UNKNOWN"
)

print("Movement classification complete.")

display(
    AIS_movement[
        [
            "MMSI",
            "Timestamp",
            "speed_mps",
            "acceleration_mps2",
            "movement_state"
        ]
    ].head()
)

print("\nMovement state distribution:")

display(
    AIS_movement["movement_state"]
        .value_counts(dropna=False)
)

In [ ]:
print("Final dataset shape:")
display(AIS_movement.shape)

print("\nNull values summary:")

display(
    AIS_movement.isna()
        .sum()
        .sort_values(ascending=False)
)

In [ ]:
metadata_cols = [
    "MMSI",
    "VesselName",
    "CallSign",
    "IMO",
    "VesselType",
    "Length",
    "Width",
    "Draft",
    "TransceiverClass"
]

In [ ]:
metadata_source = AIS_cleaned[metadata_cols].copy()

metadata_source["completeness_score"] = (
    metadata_source
        .drop(columns=["MMSI"])
        .notna()
        .sum(axis=1)
)

In [ ]:
metadata_lookup = (
    metadata_source
    .sort_values(
        by=["MMSI", "completeness_score"],
        ascending=[True, False]
    )
    .drop_duplicates(subset=["MMSI"])
    .drop(columns=["completeness_score"])
)
AIS_behavioural = (
    AIS_movement
    .merge(
        metadata_lookup,
        on="MMSI",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
print("AIS_behavioural shape:", AIS_behavioural.shape)
print("Columns:", AIS_behavioural.columns.tolist())
print("Missing vessel_id check:", AIS_behavioural["MMSI"].isna().sum(), "null MMSIs")


In [ ]:
output_path = PROCESSED_DIR / "AIS_2024_01_panynj_behavioural.parquet"

AIS_behavioural.to_parquet(
    output_path,
    index=False
)

print("Behavioural dataset exported to:")
print(output_path)

print("Rows exported:", len(AIS_behavioural))